# 1、LangSmith概述
## 1.1 什么是LangSmith?
LangSmith 是 LangChain 生态系统中专门用于 LLM（大语言模型）应用调试、监控、评估和管理 的平台。
*  追踪(tracing)：记录每次 LLM 调用的详细信息
*  监控(monitoring)：实时查看应用性能
*  调试(debug)：排查问题和优化性能
*  评估(evaluate)：系统化测试 LLM 应用


In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

print(model.invoke("你好,请介绍一下你自己"))


content='你好呀！很高兴认识你！😊\n\n我是 **DeepSeek**，一个由深度求索公司创造的AI助手。让我简单介绍一下自己：\n\n## 我的基本特点\n\n- **身份**：AI智能助手，专注于理解和生成文字内容\n- **知识截止**：2026年2月\n- **版本**：DeepSeek最新版模型\n\n## 我能做什么\n\n✅ **文本处理**：回答问题、写作、翻译、编程、分析等\n✅ **文件阅读**：支持上传图像、TXT、PDF、PPT、Word、Excel文件并提取文字信息\n✅ **图片识别**：可以接收你上传的图片，识别和分析其中的可见信息\n✅ **长文本处理**：上下文可达1M，能一次性处理《三体》三部曲这样的长篇内容\n✅ **联网搜索**：支持联网功能（需要你在Web/App手动开启）\n✅ **语音输入**：App端支持语音交互\n\n## 我的优势\n\n- 🆓 **完全免费**：没有任何收费计划\n- 📱 **多平台**：Web端和App端都可以使用\n- 🎯 **专注实用**：用热情、细腻的方式提供有帮助的回答\n\n有什么我可以帮你的吗？无论是学习、工作还是生活上的问题，随时都可以问我！' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户让我介绍一下自己。这是一个非常常见且简单的开场问题。用户可能是第一次接触我，想了解我的基本身份和能力，以便后续更好地使用我。\n\n我需要给出一个清晰、友好、全面的自我介绍。应该涵盖我的身份、核心功能、关键特点，并保持热情和乐于助人的语气。想到了可以从问候开始，然后说明我是谁、由谁创造，接着分块介绍我的主要能力，比如文本处理、文件支持、上下文长度、联网搜索和语音输入等，最后强调免费和联系方式，并以开放性问题结束，邀请用户提出具体需求。\n\n回复结构可以这样：先问候并表明身份，然后用概括性语言介绍核心能力，再以要点形式列出关键特点，最后表达服务意愿并引导对话继续。注意避免技术性过强，保持信息易懂且有帮助。'} response_metadata={'token_usage': {'completion_tokens': 440, 'prompt_tokens': 35, 'total_tokens': 47

在配置文件.env中添加以下内容：
```
LANGSMITH_API_KEY=
LANGSMITH_ENDPOINT=
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=
```
即可正常使用。在WEB页面查询AGENT运行记录。


In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

messages = [
    "请用一句话介绍一下你自己",
    "中国的首都是哪里?",
    "日本的首相是谁?",
]

results = model.batch(messages, config={"max_concurrency": 2})
print(type(results))
for question, result in zip(messages, results):
    print(f"问题：{question}")
    print(f"回答：{result.text}\n")

<class 'list'>
问题：请用一句话介绍一下你自己
回答：我是 Codex，一个协助你编写、调试和改进代码的 AI 助手。

问题：中国的首都是哪里?
回答：中国的首都是北京。

问题：日本的首相是谁?
回答：截至2026年6月，日本首相是高市早苗。



In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    configurable_fields=("model", "temperature"),  # 允许在调用时修改 model 和 temperature
)
question = "请回答:当前日本的首相是谁？"

response1 = model.invoke(question)  # 使用默认的 deepseek-v4-flash
response2 = model.invoke(
    question,
    config={"configurable": {"model": "deepseek:deepseek-v4-pro"}},  # 本次调用换成 deepseek-v4-pro
)
print(f"默认模型：{response1.response_metadata['model_name']}，回答：{response1.content}")
print(f"切换后：{response2.response_metadata['model_name']}，回答：{response2.content}")

默认模型：deepseek-flash，回答：当前日本首相（内阁总理大臣）是**高市早苗**。她于2025年10月21日就任，是日本首位女性首相。
切换后：deepseek-v4-pro，回答：截至2026年4月，日本首相是 **石破茂**。


## 1.2 具体功能
功能1：核心应用与开发
1. Tracing（追踪）
    * 功能：这是 LangSmith 最核心的功能。它会完整记录你大模型应用的每一次调用链路（Trace）。
    * 作用：当你的 Agent（智能体）或 RAG 系统运行变慢或报错时，点击进入对应的项目（如上图中的 langchain1.2_smith ），你可以看到每一步具体的 Prompt 是什么、模型返回了什么、消耗了多少 Token，以及每一个链条节点的耗时，非常方便排查 Bug 和优化性能。

2. Monitoring（监控）
    * 功能：提供生产环境的高级数据可视化看板。
    * 作用：帮你从宏观角度监控应用在一段时间内的运行状况。你可以看到 Token 消耗趋势、QPS（每秒请求数）、错误率、平均延迟（Latency）以及成本预估。适合应用上线后观察系统的稳定性和开销。

3. Datasets & Experiments（数据集与实验）
    * 功能：用于管理测试数据集并运行对比实验。
    * 作用：你可以把用户的真实输入、特定的边界情况（Edge Cases）存为数据集。当你修改了Prompt 或更换了底层大模型时，可以在这里运行自动化对比测试，直观看到新旧版本在同一批测试集上的表现差异。

4. Evaluators（评估器）
    * 功能：配置和自动化评估任务。
    * 作用：大模型的输出往往难以用传统的断言（Assert）来测试。这里允许你配置基于规则（如关键词匹配）或基于模型（LLM-as-a-judge）的评估指标（如：答案相关性、是否包含幻觉等），对追踪到的数据或实验结果进行自动打分。